In [17]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, LongType, FloatType, StringType, StructType, StructField # Added StringType
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os 

In [18]:
import pyspark
print(pyspark.__version__)

3.2.1


In [19]:
# Set Python version for both driver and workers
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (
    SparkSession.builder
    .appName("Assignment--ML-LogisticRegression")
    .master("spark://spark-master:7077")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)

In [20]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "100000")

In [21]:
temp_df = spark.read.csv('hdfs://namenode:9000/mydata/HC_application_train.csv', header=True, inferSchema=True)
print("Inferred Schema:")
temp_df.printSchema()

Inferred Schema:
root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- CODE_GENDER: string (nullable = true)
 |-- FLAG_OWN_CAR: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- CNT_CHILDREN: integer (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- NAME_TYPE_SUITE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- REGION_POPULATION_RELATIVE: double (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)
 |-- DAYS_REGISTRATION: double (nullable = true)
 |-- DAYS_ID_PUBLISH: integer (nullable = true

In [22]:
# Identify Categorical and Numeric Features (excluding ID/Label)
id_label_cols = ['SK_ID_CURR', 'TARGET', 'label'] # Include potential 'label' if renamed
all_cols = temp_df.columns
categorical_cols = []
numeric_cols = []
schema_fields = []

for field in temp_df.schema.fields:
    col_name = field.name
    col_type = field.dataType

    if col_name in id_label_cols:
        schema_fields.append(StructField(col_name, IntegerType(), True)) # Keep ID/Label as Integer
        continue # Skip adding to feature lists

    if isinstance(col_type, StringType):
        categorical_cols.append(col_name)
        schema_fields.append(StructField(col_name, StringType(), True))
    elif isinstance(col_type, (DoubleType, IntegerType, LongType, FloatType)):
        numeric_cols.append(col_name)
        schema_fields.append(StructField(col_name, DoubleType(), True)) # Load all numeric as Double for MLlib
    else:
        print(f"Warning: Column '{col_name}' has unhandled type {col_type}. Skipping.")

print(f"\nIdentified {len(numeric_cols)} numeric feature columns.")
print(f"Identified {len(categorical_cols)} categorical feature columns.")


Identified 104 numeric feature columns.
Identified 16 categorical feature columns.


In [23]:
defined_schema = StructType(schema_fields)

# Load data with defined schema
df = spark.read.csv('hdfs://namenode:9000/mydata/HC_application_train.csv', header=True, schema=defined_schema)

# Rename TARGET to label (required by MLlib)
df = df.withColumnRenamed("TARGET", "label")

In [24]:
# Basic Data Inspection
print(f"DataFrame Schema:")
df.printSchema()
print(f"Number of rows: {df.count()}")
# df.describe().show() # Can be computationally expensive

# Check for nulls (example for a few columns)
# print("Null counts:")
# df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns[:10]]).show() # Check first 10

# Check for duplicates (can be expensive)
# duplicate_count = df.groupBy(df.columns).count().where(F.col("count") > 1).count()
# print(f"Number of duplicate rows: {duplicate_count}")

DataFrame Schema:
root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- label: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- CODE_GENDER: string (nullable = true)
 |-- FLAG_OWN_CAR: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- CNT_CHILDREN: double (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- NAME_TYPE_SUITE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- REGION_POPULATION_RELATIVE: double (nullable = true)
 |-- DAYS_BIRTH: double (nullable = true)
 |-- DAYS_EMPLOYED: double (nullable = true)
 |-- DAYS_REGISTRATION: double (nullable = true)
 |-- DAYS_ID_PUBLISH: double (nullable = true)
 |

In [25]:
# 3. Initial Data Analysis (Optional in PySpark context, mirroring notebook)
print("Target Distribution:")
df.groupBy("label").count().show()
total_count = df.count()
target_distribution = df.groupBy("label").count().withColumn("proportion", F.col("count") / total_count)
target_distribution.show()

initial_default_rate = target_distribution.filter(F.col("label") == 1).first()["proportion"]
print(f"Initial Default Rate: {initial_default_rate:.2%}")

Target Distribution:
+-----+------+
|label| count|
+-----+------+
|    1| 24825|
|    0|282686|
+-----+------+

+-----+------+-------------------+
|label| count|         proportion|
+-----+------+-------------------+
|    1| 24825|0.08072881945686496|
|    0|282686| 0.9192711805431351|
+-----+------+-------------------+

Initial Default Rate: 8.07%


In [26]:
# 4. Handling Outliers using IQR (Applied *after* Train/Test Split)

# Function to calculate IQR bounds on training data
def get_iqr_bounds(df_train, columns):
    bounds = {}
    print(f"Calculating IQR bounds for {len(columns)} columns...")
    for i, col_name in enumerate(columns):
        print(f"Processing column {i+1}/{len(columns)}: {col_name}")
        try:
            # Check for non-null count first (more reliable than just trying)
            non_null_count = df_train.select(col_name).na.drop().count()
            if non_null_count == 0:
                print(f"  --> Skipping column '{col_name}' as it contains only NULL values.")
                # Decide how to handle: skip or add default bounds (e.g., None)
                # Skipping for now:
                continue

            quantiles = df_train.approxQuantile(col_name, [0.25, 0.75], 0.01) # 0.01 relative error

            # Add safety check for the returned list length
            if len(quantiles) < 2:
                 print(f"  --> Warning: approxQuantile for column '{col_name}' returned an unexpected list: {quantiles}. Skipping.")
                 continue
            elif quantiles[0] is None or quantiles[1] is None:
                 print(f"  --> Warning: approxQuantile for column '{col_name}' returned None values in quantiles: {quantiles}. Skipping.")
                 continue


            q1 = quantiles[0]
            q3 = quantiles[1]

            # Check if q1 and q3 are valid numbers (handle potential None if safety check above is removed)
            if q1 is None or q3 is None:
                 print(f"  --> Warning: Quantile calculation resulted in None for column '{col_name}'. Skipping.")
                 continue

            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr
            bounds[col_name] = (lower_bound, upper_bound)
            # Optional: print bounds for verification
            # print(f"  IQR Bounds for {col_name}: ({lower_bound:.2f}, {upper_bound:.2f})")

        except Exception as e:
            # Catch any other unexpected errors during quantile calculation for a specific column
            print(f"  --> Error calculating quantiles for column '{col_name}': {e}. Skipping.")
            continue # Skip this column and proceed to the next

    print(f"Finished calculating IQR bounds. Found bounds for {len(bounds)} columns.")
    return bounds

In [27]:
def cap_outliers(df, bounds_dict):
    df_capped = df
    # Check if bounds_dict is not empty
    if not bounds_dict:
         print("Warning: Bounds dictionary is empty. No outlier capping applied.")
         return df_capped
         
    for col_name, bounds in bounds_dict.items():
        # Ensure bounds tuple is valid
        if bounds is None or len(bounds) != 2:
             print(f"Warning: Invalid bounds for column '{col_name}'. Skipping capping for this column.")
             continue
             
        lower, upper = bounds
        # Ensure lower and upper bounds are valid numbers
        if lower is None or upper is None:
             print(f"Warning: None value encountered in bounds for column '{col_name}'. Skipping capping for this column.")
             continue

        # Check if the column exists in the DataFrame before applying capping
        if col_name in df.columns:
             df_capped = df_capped.withColumn(
                 col_name,
                 F.when(F.col(col_name).isNull(), F.col(col_name)) # Preserve nulls
                  .when(F.col(col_name) < lower, lower)
                  .when(F.col(col_name) > upper, upper)
                  .otherwise(F.col(col_name))
             )
        else:
             print(f"Warning: Column '{col_name}' not found in DataFrame during capping. Skipping.")
    return df_capped

In [28]:
# 5. Train/Test Split (Same as before)
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f"Train set size: {train_df.count()}")
print(f"Test set size: {test_df.count()}")


# --- Apply IQR Capping ---
# Use the *identified* numeric_cols list
print(f"Calculating IQR bounds on Training Data for {len(numeric_cols)} numeric columns...")
iqr_bounds = get_iqr_bounds(train_df, numeric_cols) # Use identified numeric columns

print("Applying IQR capping to Train Data...")
train_df_capped = cap_outliers(train_df, iqr_bounds)
print("Applying IQR capping to Test Data...")
test_df_capped = cap_outliers(test_df, iqr_bounds) # Use same bounds from train

train_df_capped.cache()
test_df_capped.cache()

Train set size: 245966
Test set size: 61545
Calculating IQR bounds on Training Data for 104 numeric columns...
Calculating IQR bounds for 104 columns...
Processing column 1/104: CNT_CHILDREN
Processing column 2/104: AMT_INCOME_TOTAL
Processing column 3/104: AMT_CREDIT
Processing column 4/104: AMT_ANNUITY
Processing column 5/104: AMT_GOODS_PRICE
Processing column 6/104: REGION_POPULATION_RELATIVE
Processing column 7/104: DAYS_BIRTH
Processing column 8/104: DAYS_EMPLOYED
Processing column 9/104: DAYS_REGISTRATION
Processing column 10/104: DAYS_ID_PUBLISH
Processing column 11/104: OWN_CAR_AGE
Processing column 12/104: FLAG_MOBIL
Processing column 13/104: FLAG_EMP_PHONE
Processing column 14/104: FLAG_WORK_PHONE
Processing column 15/104: FLAG_CONT_MOBILE
Processing column 16/104: FLAG_PHONE
Processing column 17/104: FLAG_EMAIL
Processing column 18/104: CNT_FAM_MEMBERS
Processing column 19/104: REGION_RATING_CLIENT
Processing column 20/104: REGION_RATING_CLIENT_W_CITY
Processing column 21/10

DataFrame[SK_ID_CURR: int, label: int, NAME_CONTRACT_TYPE: string, CODE_GENDER: string, FLAG_OWN_CAR: string, FLAG_OWN_REALTY: string, CNT_CHILDREN: double, AMT_INCOME_TOTAL: double, AMT_CREDIT: double, AMT_ANNUITY: double, AMT_GOODS_PRICE: double, NAME_TYPE_SUITE: string, NAME_INCOME_TYPE: string, NAME_EDUCATION_TYPE: string, NAME_FAMILY_STATUS: string, NAME_HOUSING_TYPE: string, REGION_POPULATION_RELATIVE: double, DAYS_BIRTH: double, DAYS_EMPLOYED: double, DAYS_REGISTRATION: double, DAYS_ID_PUBLISH: double, OWN_CAR_AGE: double, FLAG_MOBIL: double, FLAG_EMP_PHONE: double, FLAG_WORK_PHONE: double, FLAG_CONT_MOBILE: double, FLAG_PHONE: double, FLAG_EMAIL: double, OCCUPATION_TYPE: string, CNT_FAM_MEMBERS: double, REGION_RATING_CLIENT: double, REGION_RATING_CLIENT_W_CITY: double, WEEKDAY_APPR_PROCESS_START: string, HOUR_APPR_PROCESS_START: double, REG_REGION_NOT_LIVE_REGION: double, REG_REGION_NOT_WORK_REGION: double, LIVE_REGION_NOT_WORK_REGION: double, REG_CITY_NOT_LIVE_CITY: double, RE

In [29]:
# 6. Feature Engineering Pipeline Stages

# Stage 1: String Indexing for Categorical Columns
indexers = [
    StringIndexer(inputCol=col, outputCol=col + "_index", handleInvalid="skip")
    for col in categorical_cols
]

# Stage 2: One-Hot Encoding for Indexed Categorical Columns
# Note: If you have many categories, consider hashing or other techniques
# OHE creates sparse vectors, which VectorAssembler handles
encoder_input_cols = [col + "_index" for col in categorical_cols]
encoder_output_cols = [col + "_vec" for col in categorical_cols]
encoder = OneHotEncoder(inputCols=encoder_input_cols, outputCols=encoder_output_cols)

# Stage 3: Assemble Features
# Combine *capped* numeric features and *encoded* categorical features
assembler_input_cols = numeric_cols + encoder_output_cols # Use capped numeric cols + OHE output cols
assembler = VectorAssembler(inputCols=assembler_input_cols, outputCol="features", handleInvalid="skip")

# Stages list for the pipeline
preprocessing_stages = indexers + [encoder, assembler]

In [30]:
# 7. Handling Imbalance - Class Weighting (Optimal PySpark approach)
# Calculate weights: weight = total_samples / (num_classes * num_samples_in_class)
num_neg = train_df_capped.filter(F.col("label") == 0).count()
num_pos = train_df_capped.filter(F.col("label") == 1).count()
total_train = train_df_capped.count()
num_classes = 2
weight_neg = total_train / (num_classes * num_neg) if num_neg > 0 else 0
weight_pos = total_train / (num_classes * num_pos) if num_pos > 0 else 0
print(f"Weight for class 0: {weight_neg:.2f}")
print(f"Weight for class 1: {weight_pos:.2f}")

# Add weight column to the training dataframe *before* passing to CV
train_df_weighted = train_df_capped.withColumn(
    "weight",
    F.when(F.col("label") == 1, weight_pos).otherwise(weight_neg)
)
train_df_weighted.cache() # Cache this one as it's used for fitting

# --- Alternative: Undersampling (Manual Implementation) ---
# fraction_neg = num_pos / num_neg
# train_df_neg_sampled = train_df_capped.filter(F.col("label") == 0).sample(False, fraction_neg, seed=42)
# train_df_pos = train_df_capped.filter(F.col("label") == 1)
# train_df_undersampled = train_df_neg_sampled.unionAll(train_df_pos)
# print("Train set size after undersampling:", train_df_undersampled.count())
# print("Target distribution after undersampling:")
# train_df_undersampled.groupBy("label").count().show()

# --- Alternative: SMOTE (Difficult in PySpark, usually avoided) ---
# No direct SMOTE implementation. Would typically use weighting or simpler resampling.

# --- Alternative: Raw Data (No IQR, No Weighting/SMOTE) ---
# Use train_df directly for training below, without capping and without weightCol

Weight for class 0: 0.54
Weight for class 1: 6.20


DataFrame[SK_ID_CURR: int, label: int, NAME_CONTRACT_TYPE: string, CODE_GENDER: string, FLAG_OWN_CAR: string, FLAG_OWN_REALTY: string, CNT_CHILDREN: double, AMT_INCOME_TOTAL: double, AMT_CREDIT: double, AMT_ANNUITY: double, AMT_GOODS_PRICE: double, NAME_TYPE_SUITE: string, NAME_INCOME_TYPE: string, NAME_EDUCATION_TYPE: string, NAME_FAMILY_STATUS: string, NAME_HOUSING_TYPE: string, REGION_POPULATION_RELATIVE: double, DAYS_BIRTH: double, DAYS_EMPLOYED: double, DAYS_REGISTRATION: double, DAYS_ID_PUBLISH: double, OWN_CAR_AGE: double, FLAG_MOBIL: double, FLAG_EMP_PHONE: double, FLAG_WORK_PHONE: double, FLAG_CONT_MOBILE: double, FLAG_PHONE: double, FLAG_EMAIL: double, OCCUPATION_TYPE: string, CNT_FAM_MEMBERS: double, REGION_RATING_CLIENT: double, REGION_RATING_CLIENT_W_CITY: double, WEEKDAY_APPR_PROCESS_START: string, HOUR_APPR_PROCESS_START: double, REG_REGION_NOT_LIVE_REGION: double, REG_REGION_NOT_WORK_REGION: double, LIVE_REGION_NOT_WORK_REGION: double, REG_CITY_NOT_LIVE_CITY: double, RE

In [31]:
# 8. Define Logistic Regression Estimator (with weighting)
lr = LogisticRegression(featuresCol="features", labelCol="label", weightCol="weight", maxIter=1000)

In [33]:
# 9. Hyperparameter Tuning Setup

# Define the pipeline *including preprocessing*
pipeline = Pipeline(stages=preprocessing_stages + [lr])

# Define ParamGrid using the pipeline stages
# Example: Tuning LR's regParam and elasticNetParam, and potentially assembler's handleInvalid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()
    # NOTE: Your previous run found best params: {'C': 10, 'max_iter': 100, 'penalty': 'l1', 'solver': 'liblinear'}
    # This translates roughly to Spark params: regParam=0.1 (1/C), elasticNetParam=1.0 (L1)
    # Let's redefine the grid around these values if desired, or use the best_estimator directly if skipping CV again.
    # For demonstration, let's run CV with a refined grid:
paramGrid_refined = ParamGridBuilder().addGrid(lr.regParam, [0.01, 0.1, 0.5]).addGrid(lr.elasticNetParam, [1.0]).build()
    # Using liblinear directly isn't possible, saga might be closer if needed, but L-BFGS is default
    # max_iter can also be tuned if needed, but 1000 is often sufficient

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

cv = CrossValidator(estimator=pipeline, # Use the full pipeline
                    estimatorParamMaps=paramGrid_refined, # Use the refined grid
                    evaluator=evaluator,
                    numFolds=3,
                    parallelism=4,
                    seed=42)

In [34]:
# 10. Train Model using Cross-Validation
print("Starting Hyperparameter Tuning with Cross-Validation (with preprocessing pipeline)...")
# Fit on the *weighted* training data (input to the full pipeline)
cvModel = cv.fit(train_df_weighted) # The CV object uses the pipeline defined in its estimator

print("Finished Tuning.")
bestPipelineModel = cvModel.bestModel # The best estimator is the entire PipelineModel

# Extract the best Logistic Regression model stage for inspection
lrModel = bestPipelineModel.stages[-1]

print(f"Best Logistic Regression Parameters from CV:")
print(f"  regParam: {lrModel.getRegParam()}")
print(f"  elasticNetParam: {lrModel.getElasticNetParam()}")
# print best CV score
avgMetrics = cvModel.avgMetrics
bestMetric = evaluator.getMetricName()
bestScore = max(avgMetrics)
print(f"Best Cross-validation score ({bestMetric}): {bestScore:.4f}")

Starting Hyperparameter Tuning with Cross-Validation (with preprocessing pipeline)...
Finished Tuning.
Best Logistic Regression Parameters from CV:
  regParam: 0.01
  elasticNetParam: 1.0
Best Cross-validation score (areaUnderROC): 0.7168


In [35]:
# 11. Make Predictions (using the best *PipelineModel*)
print("Making predictions on Test set...")
# The pipeline model handles all steps: index/encode -> assemble -> predict
predictions_test = bestPipelineModel.transform(test_df_capped) # Apply to capped test data

print("Making predictions on Train set (resampled for consistent evaluation)...")
# To evaluate on the training data used for tuning, we need to apply the pipeline
# to the *weighted* training data.
predictions_train = bestPipelineModel.transform(train_df_weighted)


predictions_test.cache()
predictions_train.cache()

Making predictions on Test set...
Making predictions on Train set (resampled for consistent evaluation)...


DataFrame[SK_ID_CURR: int, label: int, NAME_CONTRACT_TYPE: string, CODE_GENDER: string, FLAG_OWN_CAR: string, FLAG_OWN_REALTY: string, CNT_CHILDREN: double, AMT_INCOME_TOTAL: double, AMT_CREDIT: double, AMT_ANNUITY: double, AMT_GOODS_PRICE: double, NAME_TYPE_SUITE: string, NAME_INCOME_TYPE: string, NAME_EDUCATION_TYPE: string, NAME_FAMILY_STATUS: string, NAME_HOUSING_TYPE: string, REGION_POPULATION_RELATIVE: double, DAYS_BIRTH: double, DAYS_EMPLOYED: double, DAYS_REGISTRATION: double, DAYS_ID_PUBLISH: double, OWN_CAR_AGE: double, FLAG_MOBIL: double, FLAG_EMP_PHONE: double, FLAG_WORK_PHONE: double, FLAG_CONT_MOBILE: double, FLAG_PHONE: double, FLAG_EMAIL: double, OCCUPATION_TYPE: string, CNT_FAM_MEMBERS: double, REGION_RATING_CLIENT: double, REGION_RATING_CLIENT_W_CITY: double, WEEKDAY_APPR_PROCESS_START: string, HOUR_APPR_PROCESS_START: double, REG_REGION_NOT_LIVE_REGION: double, REG_REGION_NOT_WORK_REGION: double, LIVE_REGION_NOT_WORK_REGION: double, REG_CITY_NOT_LIVE_CITY: double, RE

In [37]:
# 12. Evaluate Model

print("\n--- Model Evaluation (After IQR, Weighting, Tuning) ---")

# ROC AUC (BinaryClassificationEvaluator is fine with integer labels)
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc_test = evaluator.evaluate(predictions_test)
auc_train = evaluator.evaluate(predictions_train)
print(f"ROC-AUC (Train-Proba)   : {auc_train:.2f}")
print(f"ROC-AUC (Test-Proba)    : {auc_test:.2f}")

# Other Metrics using MulticlassClassificationEvaluator
# Ensure BOTH label and prediction columns are DoubleType for this evaluator
print("Casting label and prediction to DoubleType for MulticlassClassificationEvaluator...")
predictions_test_eval = predictions_test.withColumn("label", F.col("label").cast(DoubleType())) \
                                        .withColumn("prediction", F.col("prediction").cast(DoubleType()))
predictions_train_eval = predictions_train.withColumn("label", F.col("label").cast(DoubleType())) \
                                          .withColumn("prediction", F.col("prediction").cast(DoubleType()))

# Initialize evaluator (label and prediction cols are now DoubleType)
multi_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

# Evaluate metrics
acc_test = multi_evaluator.evaluate(predictions_test_eval, {multi_evaluator.metricName: "accuracy"})
f1_test = multi_evaluator.evaluate(predictions_test_eval, {multi_evaluator.metricName: "f1"})
# Use 1.0 for the label when requesting precision/recall by label
prec_test = multi_evaluator.evaluate(predictions_test_eval, {multi_evaluator.metricName: "precisionByLabel", multi_evaluator.metricLabel: 1.0})
rec_test = multi_evaluator.evaluate(predictions_test_eval, {multi_evaluator.metricLabel: 1.0, multi_evaluator.metricName: "recallByLabel"})

acc_train = multi_evaluator.evaluate(predictions_train_eval, {multi_evaluator.metricName: "accuracy"})
f1_train = multi_evaluator.evaluate(predictions_train_eval, {multi_evaluator.metricName: "f1"})
prec_train = multi_evaluator.evaluate(predictions_train_eval, {multi_evaluator.metricName: "precisionByLabel", multi_evaluator.metricLabel: 1.0})
rec_train = multi_evaluator.evaluate(predictions_train_eval, {multi_evaluator.metricLabel: 1.0, multi_evaluator.metricName: "recallByLabel"})

# Print evaluation metrics
print(f"Accuracy (Train Set)    : {acc_train:.2f}")
print(f"Accuracy (Test Set)     : {acc_test:.2f}")
print(f"Precision (Pos Class 1 - Train): {prec_train:.2f}")
print(f"Precision (Pos Class 1 - Test) : {prec_test:.2f}")
print(f"Recall (Pos Class 1 - Train)   : {rec_train:.2f}")
print(f"Recall (Pos Class 1 - Test)    : {rec_test:.2f}")
print(f"F1-Score (Train Set)    : {f1_train:.2f}")
print(f"F1-Score (Test Set)     : {f1_test:.2f}")

# (Rest of the code for Confusion Matrix, Feature Importance, etc.)


--- Model Evaluation (After IQR, Weighting, Tuning) ---
ROC-AUC (Train-Proba)   : 0.78
ROC-AUC (Test-Proba)    : 0.77
Casting label and prediction to DoubleType for MulticlassClassificationEvaluator...
Accuracy (Train Set)    : 0.79
Accuracy (Test Set)     : 0.78
Precision (Pos Class 1 - Train): 0.16
Precision (Pos Class 1 - Test) : 0.16
Recall (Pos Class 1 - Train)   : 0.59
Recall (Pos Class 1 - Test)    : 0.56
F1-Score (Train Set)    : 0.84
F1-Score (Test Set)     : 0.83


In [38]:
# 13. Confusion Matrix (Manual Calculation)
print("\nConfusion Matrix (Test Set):")
conf_matrix_df = predictions_test.select("label", "prediction").groupBy("label", "prediction").count()
conf_matrix_df.show()

# For a more standard matrix display, collect results (only safe for smaller test sets)
try:
    cm_pd = conf_matrix_df.toPandas()
    # Pivot to get the matrix format (assuming labels 0 and 1)
    cm_pivot = cm_pd.pivot(index='label', columns='prediction', values='count').fillna(0)
    print("Confusion Matrix (Test Set - Pandas):\n", cm_pivot)

    # Display using ConfusionMatrixDisplay if sklearn is available in the environment
    try:
        from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
        # Need actual y_test and y_pred collected
        y_test_collected = predictions_test.select("label").rdd.flatMap(lambda x: x).collect()
        y_pred_collected = predictions_test.select("prediction").rdd.flatMap(lambda x: x).collect()
        cm = confusion_matrix(y_test_collected, y_pred_collected)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot(cmap=plt.cm.Blues)
        plt.title('Confusion Matrix - PySpark Test Data')
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("Scikit-learn not available for ConfusionMatrixDisplay plot.")
except Exception as e:
    print(f"Could not collect confusion matrix to Pandas: {e}")


Confusion Matrix (Test Set):
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0|   47|
|    0|       0.0| 1257|
|    1|       1.0|   61|
|    0|       1.0|  326|
+-----+----------+-----+

Confusion Matrix (Test Set - Pandas):
 prediction   0.0  1.0
label                
0           1257  326
1             47   61
Could not collect confusion matrix to Pandas: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 2081.0 failed 4 times, most recent failure: Lost task 1.3 in stage 2081.0 (TID 2857) (172.18.0.8 executor 0): java.io.IOException: Cannot run program "/opt/conda/bin/python": error=2, No such file or directory
	at java.lang.ProcessBuilder.start(ProcessBuilder.java:1048)
	at org.apache.spark.api.python.PythonWorkerFactory.startDaemon(PythonWorkerFactory.scala:215)
	at org.apache.spark.api.python.PythonWorkerFactory.crea

In [40]:
# 14. Feature Importance
print("\n--- Feature Importance (Best Model) ---")
lr_model_stage = bestPipelineModel.stages[-1] # LR is the last stage
assembler_stage = bestPipelineModel.stages[-2] # Assembler is second to last

# Get feature names from the assembler's metadata or inputCols
# Using assembler_stage.getInputCols() is more robust if metadata isn't set properly
feature_names = assembler_stage.getInputCols()
importances = lr_model_stage.coefficients.toArray()

if len(feature_names) != len(importances):
     print(f"Warning: Mismatch in feature names ({len(feature_names)}) and coefficients ({len(importances)})")
     # Fallback or error handling needed here
else:
    feat_importances = pd.Series(importances, index=feature_names)
    plt.figure(figsize=(10, 8))
    ax = feat_importances.abs().nlargest(25).plot(kind='barh')
    ax.invert_yaxis()
    plt.title('Feature Importance (Best Tuned Model)')
    plt.xlabel('Coefficient Value (Absolute)')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.show()


--- Feature Importance (Best Model) ---


In [41]:
# 15. Calculate Predicted Default Rate
print("\n--- Predicted Default Rate ---")
def calculate_spark_default_rate(predictions_df, dataset_name="dataset", threshold=0.5):
    # Extract probability for class 1
    extract_prob_udf = F.udf(lambda v: float(v[1]), DoubleType())
    pred_df = predictions_df.withColumn("prob_pos", extract_prob_udf(F.col("probability")))

    # Apply threshold
    pred_df = pred_df.withColumn("pred_default", (F.col("prob_pos") > threshold).cast(IntegerType()))

    # Count defaults and total
    default_count = pred_df.filter(F.col("pred_default") == 1).count()
    total_count_pred = pred_df.count()

    # Calculate rate
    default_rate = default_count / total_count_pred if total_count_pred > 0 else 0

    # Display results
    print(f"Default rate calculation for dataset {dataset_name}:")
    print(f"  Threshold used: {threshold}")
    print(f"  Number predicted as default: {default_count}")
    print(f"  Total predictions: {total_count_pred}")
    print(f"  Predicted default rate: {default_rate:.2%}")
    return default_rate

calculate_spark_default_rate(predictions_train, dataset_name="Train (SMOTE+IQR)")
calculate_spark_default_rate(predictions_test, dataset_name="Test (IQR)")


--- Predicted Default Rate ---


Py4JJavaError: An error occurred while calling o32107.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 2082.0 failed 4 times, most recent failure: Lost task 0.3 in stage 2082.0 (TID 2865) (172.18.0.8 executor 0): java.io.IOException: Cannot run program "/opt/conda/bin/python": error=2, No such file or directory
	at java.lang.ProcessBuilder.start(ProcessBuilder.java:1048)
	at org.apache.spark.api.python.PythonWorkerFactory.startDaemon(PythonWorkerFactory.scala:215)
	at org.apache.spark.api.python.PythonWorkerFactory.createThroughDaemon(PythonWorkerFactory.scala:133)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:106)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:121)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:162)
	at org.apache.spark.sql.execution.python.BatchEvalPythonExec.evaluate(BatchEvalPythonExec.scala:81)
	at org.apache.spark.sql.execution.python.Ev

In [42]:
# 16. Cleanup
train_df_capped.unpersist()
test_df_capped.unpersist()
predictions_train.unpersist()
predictions_test.unpersist()
spark.stop()